# Outlier Detection, Data Validation and Cleaning Pipelines

In [2]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "customer_id": range(101, 111),
    "age": [22, 25, 29, 31, 34, 38, 42, 45, 150, -5],
    "income": [
        25000, 30000, 35000, 40000, 45000,
        50000, 55000, 60000, 500000, -10000
    ],
    "orders": [1, 2, 3, 4, 5, 6, 7, 8, 100, -2],
    "city": [
        "Delhi", "Mumbai", "Delhi", "Pune", "Mumbai",
        "Delhi", "Pune", "Mumbai", "Delhi", "Pune"
    ]
})
df

,customer_id,age,income,orders,city
0,101,22,25000,1,Delhi
1,102,25,30000,2,Mumbai
2,103,29,35000,3,Delhi
3,104,31,40000,4,Pune
4,105,34,45000,5,Mumbai
5,106,38,50000,6,Delhi
6,107,42,55000,7,Pune
7,108,45,60000,8,Mumbai
8,109,150,500000,100,Delhi
9,110,-5,-10000,-2,Pune


#### Q1. Detect outliers using the IQR method
Calculate Q1, Q3, and the interquartile range (IQR) for income. Calculate the lower and upper bounds. Identify records outside these bounds.

In [4]:
q3=df['income'].quantile(0.75)
q1=df['income'].quantile(0.25)
lower=q1-(q3-q1)*1.5
upper=q3+(q3-q1)*1.5
df.query('income>@upper or income<@lower')

,customer_id,age,income,orders,city
8,109,150,500000,100,Delhi
9,110,-5,-10000,-2,Pune


#### Q2. Create an outlier Boolean mask
Create a Boolean mask identifying outliers in age using the IQR method. Display the outlier rows without deleting them.

In [6]:
q3=df['age'].quantile(0.75)
q1=df['age'].quantile(0.25)
lower=q1-(q3-q1)*1.5
upper=q3+(q3-q1)*1.5
mask=(df['age']>upper) | (df['age']<lower)
df.loc[mask]

,customer_id,age,income,orders,city
8,109,150,500000,100,Delhi
9,110,-5,-10000,-2,Pune


#### Q3. Validate domain constraints
Identify records with invalid values based on these business rules:
- Age must be between 0 and 120.
- Income must be non-negative.
- Orders must be non-negative.
Return all records violating at least one rule.

In [8]:
mask=((df['age'].between(0,120)) & (df['income']>0) & (df['orders']>=0))
df.loc[mask]

,customer_id,age,income,orders,city
0,101,22,25000,1,Delhi
1,102,25,30000,2,Mumbai
2,103,29,35000,3,Delhi
3,104,31,40000,4,Pune
4,105,34,45000,5,Mumbai
5,106,38,50000,6,Delhi
6,107,42,55000,7,Pune
7,108,45,60000,8,Mumbai


#### Q4. Filter invalid records
Create a new DataFrame containing only records satisfying all three domain constraints from Q3. Report how many records remain

In [10]:
clean=df.loc[mask]
clean

,customer_id,age,income,orders,city
0,101,22,25000,1,Delhi
1,102,25,30000,2,Mumbai
2,103,29,35000,3,Delhi
3,104,31,40000,4,Pune
4,105,34,45000,5,Mumbai
5,106,38,50000,6,Delhi
6,107,42,55000,7,Pune
7,108,45,60000,8,Mumbai


In [11]:
clean.shape

(8, 5)

#### Q5. Compare outlier detection methods
Detect income outliers using both the IQR method and a z-score approach.
Use the population standard deviation (ddof=0) for the z-score, and flag observations with an absolute z-score greater than 3.
Explain why the two methods may flag different records.

In [13]:
# 1. IQR method
q1 = df['income'].quantile(0.25)
q3 = df['income'].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

df['iqr_outlier'] = (
    (df['income'] < lower) |
    (df['income'] > upper)
)
# 2. Z-score method
mean_income = df['income'].mean()
std_income = df['income'].std(ddof=0)

df['z_score'] = (df['income'] - mean_income) / std_income

df['z_outlier'] = df['z_score'].abs() > 3

# 3. Display flagged records
df.loc[
    df['iqr_outlier'] | df['z_outlier'],
    ['income', 'iqr_outlier', 'z_score', 'z_outlier']
]

,income,iqr_outlier,z_score,z_outlier
8,500000,True,2.973191,False
9,-10000,True,-0.663086,False


#### Q6. Cap outliers with winsorization-style clipping
Calculate the IQR bounds for income and use clip() to cap values outside those bounds. Preserve the original income column in the original DataFrame.
Explain how clipping differs from removing outlier rows.

In [15]:
q1 = df['income'].quantile(0.25)
q3 = df['income'].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

capped_income=df['income'].copy()
capped_income=capped_income.clip(upper=upper,lower=lower)
capped_income

0    25000
1    30000
2    35000
3    40000
4    45000
5    50000
6    55000
7    60000
8    87500
9    -2500
Name: income, dtype: int64

In [16]:
print(lower,upper)

-2500.0 87500.0


#### Q7. Build a validation report
Create a report containing the number of invalid age, income, and orders values. Include the number and percentage of affected rows for each column.

In [18]:
df = pd.DataFrame({
    "customer_id": range(101, 111),
    "age": [22, 25, 29, 31, 34, 38, 42, 45, 150, -5],
    "income": [
        25000, 30000, 35000, 40000, 45000,
        50000, 55000, 60000, 500000, -10000
    ],
    "orders": [1, 2, 3, 4, 5, 6, 7, 8, 100, -2],
    "city": [
        "Delhi", "Mumbai", "Delhi", "Pune", "Mumbai",
        "Delhi", "Pune", "Mumbai", "Delhi", "Pune"
    ]
})
df
# Identify invalid values in each column
invalid_age = ~df['age'].between(0, 120)
invalid_income = df['income'] <= 0
invalid_orders = df['orders'] < 0

# Create report
report = pd.DataFrame({
    'invalid_count': [
        invalid_age.sum(),
        invalid_income.sum(),
        invalid_orders.sum()
    ],
    'affected_rows_percentage': [
        invalid_age.mean() * 100,
        invalid_income.mean() * 100,
        invalid_orders.mean() * 100
    ]
}, index=['age', 'income', 'orders'])

report['affected_rows_percentage'] = report['affected_rows_percentage'].round(2)

report

,invalid_count,affected_rows_percentage
age,2,20.0
income,1,10.0
orders,1,10.0


#### Q8. Detect suspicious values within groups
Group the data by city and calculate median income for each city. Compare each customer's income with their city's median.
Identify customers whose income is more than five times their city's median. Explain why group-based comparisons can be more informative than global thresholds.

In [20]:
city_median=df.groupby('city')['income'].transform('median')
city_median


0    42500.0
1    45000.0
2    42500.0
3    40000.0
4    45000.0
5    42500.0
6    40000.0
7    45000.0
8    42500.0
9    40000.0
Name: income, dtype: float64

In [21]:
mask=df['income']>5*city_median
df.loc[mask]

,customer_id,age,income,orders,city
8,109,150,500000,100,Delhi


#### Q9. Combine cleaning and validation
Create a cleaned DataFrame that:
1. Replaces ages outside 0–120 with missing values.
2. Replaces negative incomes and negative orders with missing values.
3. Reports missing values after cleaning.
4. Keeps all rows for auditing rather than dropping records.
5. Does not modify the original DataFrame.

In [23]:
df = pd.DataFrame({
    "customer_id": range(101, 111),
    "age": [22, 25, 29, 31, 34, 38, 42, 45, 150, -5],
    "income": [
        25000, 30000, 35000, 40000, 45000,
        50000, 55000, 60000, 500000, -10000
    ],
    "orders": [1, 2, 3, 4, 5, 6, 7, 8, 100, -2],
    "city": [
        "Delhi", "Mumbai", "Delhi", "Pune", "Mumbai",
        "Delhi", "Pune", "Mumbai", "Delhi", "Pune"
    ]
})
df

,customer_id,age,income,orders,city
0,101,22,25000,1,Delhi
1,102,25,30000,2,Mumbai
2,103,29,35000,3,Delhi
3,104,31,40000,4,Pune
4,105,34,45000,5,Mumbai
5,106,38,50000,6,Delhi
6,107,42,55000,7,Pune
7,108,45,60000,8,Mumbai
8,109,150,500000,100,Delhi
9,110,-5,-10000,-2,Pune


In [24]:
#Replaces ages outside 0–120 with missing values.
temp=df.copy()
temp.loc[~temp['age'].between(0,120),'age']=np.nan

#Replaces negative incomes and negative orders with missing values
temp.loc[temp['income']<=0,'income']=np.nan
temp

,customer_id,age,income,orders,city
0,101,22.0,25000.0,1,Delhi
1,102,25.0,30000.0,2,Mumbai
2,103,29.0,35000.0,3,Delhi
3,104,31.0,40000.0,4,Pune
4,105,34.0,45000.0,5,Mumbai
5,106,38.0,50000.0,6,Delhi
6,107,42.0,55000.0,7,Pune
7,108,45.0,60000.0,8,Mumbai
8,109,NaN,500000.0,100,Delhi
9,110,NaN,NaN,-2,Pune


In [25]:
#Reports missing values after cleaning.
temp.isnull().sum()

customer_id    0
age            2
income         1
orders         0
city           0
dtype: int64

#### Q10. Mini-project — Reusable data-quality audit
Write a function data_quality_audit(data) that returns a report for each column containing:
- Data type
- Missing-value count
- Missing-value percentage
- Number of distinct non-missing values
- Number of invalid values for age, income, and orders, based on the domain rules above

In [49]:
def data_quality_audit(data):
    df=data.copy()
    summary=pd.DataFrame(
        {'Missing-value count':df.isnull().sum(),
         'Missing-value percentage':round(df.isnull().mean()*100,2),
         'distinct non-missing values':df.nunique()
        })
    return summary

print(data_quality_audit(temp))

             Missing-value count  Missing-value percentage  \
customer_id                    0                       0.0   
age                            2                      20.0   
income                         1                      10.0   
orders                         0                       0.0   
city                           0                       0.0   

             distinct non-missing values  
customer_id                           10  
age                                    8  
income                                 9  
orders                                10  
city                                   3  
